In [1]:
!pip install ultralytics
!pip install roboflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of mkl-fft to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of mkl-random to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of mkl-umath to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 89.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 108.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 78.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import json
import copy
import os
import tempfile
import torch
import torchvision
import torchvision.transforms.v2 as transforms
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import io
import sys
import glob
import cv2
import random 

from ultralytics import YOLO
from PIL import Image
from roboflow import Roboflow
from torchvision import datasets, models
from torch.utils.data import DataLoader
from torchvision.models import EfficientNet_B0_Weights
from tqdm import tqdm
from torchvision.datasets import CocoDetection
from torchvision.transforms import ToTensor
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor, fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from torchvision.ops import box_iou

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
# rf = Roboflow(api_key="sNwZKFyXE0vHSdAqPXX0")
# project = rf.workspace("main-workspace-vcien").project("birdnest-nzpjl-inn3e")
# version = project.version(4)
# dataset = version.download("coco")


rf = Roboflow(api_key="sNwZKFyXE0vHSdAqPXX0")
project = rf.workspace("main-workspace-vcien").project("birdnest-balanced")
version = project.version(1)
dataset = version.download("coco")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to BirdNest-Balanced-1 in coco:: 100%|██████████| 1805/1805 [00:00<00:00, 7861.21it/s]


In [8]:
transform = transforms.Compose([
    transforms.ToImage(),
    transforms.ToDtype(torch.float32, scale=True),
])

train_dataset = CocoDetection(
    root="BirdNest-Balanced-1/train",
    annFile="BirdNest-Balanced-1/train/_annotations.coco.json",
    transforms=transform
)

val_dataset = CocoDetection(
    root="BirdNest-Balanced-1/valid",
    annFile="BirdNest-Balanced-1/valid/_annotations.coco.json",
    transforms=transform
)

test_dataset = CocoDetection(
    root="BirdNest-Balanced-1/test",
    annFile="BirdNest-Balanced-1/test/_annotations.coco.json",
    transforms=transform
)

# train_dataset = CocoDetection(
#     root="BirdNest-4/train",
#     annFile="BirdNest-4/train/_annotations.coco.json",
#     transforms=transform
# )

# val_dataset = CocoDetection(
#     root="BirdNest-4/valid",
#     annFile="BirdNest-4/valid/_annotations.coco.json",
#     transforms=transform
# )

# test_dataset = CocoDetection(
#     root="BirdNest-4/test",
#     annFile="BirdNest-4/test/_annotations.coco.json",
#     transforms=transform
# )

# Custom collate function for object detection
def collate_fn(batch):
    images, targets = tuple(zip(*batch))

    # Convert images to tensors and stack them
    images = [torch.as_tensor(img, dtype=torch.float32) if not isinstance(img, torch.Tensor) else img for img in images]

    # Process targets to match Faster R-CNN format
    processed_targets = []
    for target in targets:
        processed_target = {}

        boxes = []
        labels = []

        for annotation in target:
            if 'bbox' in annotation and 'category_id' in annotation:
                x, y, w, h = annotation['bbox']
                boxes.append([x, y, x + w, y + h])
                labels.append(annotation['category_id'])

                # Save image_id (same for all annotations in one image)
                if 'image_id' in annotation:
                    processed_target['image_id'] = torch.tensor(annotation['image_id'])

        # Handle empty targets
        if boxes:
            processed_target['boxes'] = torch.as_tensor(boxes, dtype=torch.float32)
            processed_target['labels'] = torch.as_tensor(labels, dtype=torch.int64)
        else:
            processed_target['boxes'] = torch.zeros((0, 4), dtype=torch.float32)
            processed_target['labels'] = torch.zeros(0, dtype=torch.int64)
            processed_target['image_id'] = torch.tensor(target[0]['image_id'])  # still add image_id

        processed_targets.append(processed_target)

    return images, processed_targets
    
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False, collate_fn=collate_fn)

loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!


In [19]:
import torch
import copy
import sys
import io
import json
import tempfile
from tqdm import tqdm
from pycocotools.cocoeval import COCOeval

class NetworkTrainer:
    def __init__(self, model, optimizer, lr_scheduler, device):
        self.model = model.to(device)
        self.optimizer = optimizer
        self.lr_scheduler = lr_scheduler
        self.device = device
        self.best_map = 0.0
        self.best_model_state = copy.deepcopy(self.model.state_dict())

    def compute_loss(self, loader):
        """
        Computes loss on a given loader (usually validation).
        """
        self.model.train() # FasterRCNN computes loss only in train mode
        total_loss = 0.0
        count = 0

        with torch.no_grad():
            for imgs, targets in loader:
                imgs = [img.to(self.device) for img in imgs]
                
                # FIXED: Only move Tensors to device, leave integers/strings alone
                targets = [{k: v.to(self.device) if isinstance(v, torch.Tensor) else v for k, v in t.items()} for t in targets]

                loss_dict = self.model(imgs, targets)
                loss = sum(loss for loss in loss_dict.values())

                total_loss += loss.item()
                count += 1

        return total_loss / count if count > 0 else 0.0

    def compute_map(self, loader, iou_type="bbox"):
        self.model.eval()
        results = []
        
        with torch.no_grad():
            for imgs, targets in loader:
                imgs = [img.to(self.device) for img in imgs]
                outputs = self.model(imgs)
                
                for output, target in zip(outputs, targets):
                    # target["image_id"] might be an int or a 0-d tensor. 
                    # If it's a tensor, .item() converts it to int. If it's already int, we use it directly.
                    if isinstance(target["image_id"], torch.Tensor):
                        image_id = int(target["image_id"].item())
                    else:
                        image_id = int(target["image_id"])

                    if "boxes" in output:
                        boxes = output["boxes"].cpu().numpy()
                        scores = output["scores"].cpu().numpy()
                        labels = output["labels"].cpu().numpy()
                        
                        for box, score, label in zip(boxes, scores, labels):
                            x1, y1, x2, y2 = box
                            w, h = x2 - x1, y2 - y1
                            results.append({
                                "image_id": image_id,
                                "category_id": int(label),
                                "bbox": [float(x1), float(y1), float(w), float(h)],
                                "score": float(score)
                            })
        
        if len(results) == 0:
            return 0.0

        # Save results as temp JSON
        with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as f:
            json.dump(results, f)
            result_file = f.name
        
        base_dataset = loader.dataset
        if isinstance(base_dataset, torch.utils.data.Subset):
            base_dataset = base_dataset.dataset   
        
        coco_gt = base_dataset.coco
        
        # Suppress COCO printing
        old_stdout = sys.stdout
        sys.stdout = io.StringIO()
        
        try:
            coco_dt = coco_gt.loadRes(result_file)
            coco_eval = COCOeval(coco_gt, coco_dt, iouType=iou_type)
            coco_eval.evaluate()
            coco_eval.accumulate()
            coco_eval.summarize()
            metric = coco_eval.stats[0] # mAP@50-95
        except Exception:
            metric = 0.0
            
        sys.stdout = old_stdout 
        
        return metric

    def fit(self, train_loader, val_loader, num_epochs=30):
        print(f"Starting training on {self.device}...")
        
        for epoch in range(num_epochs):
            self.model.train()
            running_loss = 0.0
            num_batches = len(train_loader)

            pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=True)

            for imgs, targets in pbar:
                imgs = [img.to(self.device) for img in imgs]
                
                # FIXED: Only move Tensors to device
                targets = [{k: v.to(self.device) if isinstance(v, torch.Tensor) else v for k, v in t.items()} for t in targets]

                # Forward
                loss_dict = self.model(imgs, targets)
                losses = sum(loss for loss in loss_dict.values())

                # Backward
                self.optimizer.zero_grad()
                losses.backward()
                self.optimizer.step()

                # Accumulate loss
                loss_val = losses.item()
                running_loss += loss_val
                
                # Update progress bar
                pbar.set_postfix({"Batch Loss": f"{loss_val:.4f}"})

            # Step scheduler
            self.lr_scheduler.step()

            # Calculate metrics
            avg_train_loss = running_loss / num_batches
            val_loss = self.compute_loss(val_loader)
            val_map = self.compute_map(val_loader)

            print(f"Epoch {epoch+1}/{num_epochs} - "
                  f"Train Loss: {avg_train_loss:.4f} | "
                  f"Val Loss: {val_loss:.4f} | "
                  f"Val mAP@50-95: {val_map:.4f}")

            # Save best model
            if val_map > self.best_map:
                self.best_map = val_map
                self.best_model_state = copy.deepcopy(self.model.state_dict())
                torch.save(self.model.state_dict(), "best_fasterrcnn.pth")
                print(f"✅ Best model saved at epoch {epoch+1} with mAP {self.best_map:.4f}")

        self.model.load_state_dict(self.best_model_state)
        print(f"Training Complete. Loaded best model with mAP: {self.best_map:.4f}")

In [20]:
from torch.utils.data import Subset

small_train_dataset = Subset(train_dataset, list(range(20)))
small_val_dataset = Subset(val_dataset, list(range(10)))

small_train_loader = DataLoader(small_train_dataset, batch_size=2, shuffle=True, collate_fn=collate_fn)
small_val_loader = DataLoader(small_val_dataset, batch_size=2, shuffle=False, collate_fn=collate_fn)

In [21]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
weights = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
model = fasterrcnn_resnet50_fpn_v2(weights=weights)

NUM_CLASSES = 4  # A, B, C + background
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)

# params = [p for p in model.parameters() if p.requires_grad]
# optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
# lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

# trainer = NetworkTrainer(model, optimizer, lr_scheduler, device)
# trainer.fit(train_loader, val_loader, 30) 
# trainer.fit(small_train_loader, small_val_loader, 30) 


params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

trainer = NetworkTrainer(model, optimizer,  lr_scheduler, device)
trainer.fit(train_loader, val_loader, 30)
# trainer.fit(small_train_loader, small_val_loader, 5)

Starting training on cuda...


Epoch 1/30: 100%|██████████| 368/368 [07:34<00:00,  1.24s/it, Batch Loss=0.0771]


Epoch 1/30 - Train Loss: 0.1064 | Val Loss: 0.0690 | Val mAP@50-95: 0.0000


Epoch 2/30:  36%|███▌      | 131/368 [02:43<04:55,  1.25s/it, Batch Loss=0.0602]


KeyboardInterrupt: 

In [ ]:
import shutil
import os
from IPython.display import FileLink

# 1. Configuration
# This must match the name you used in torch.save() inside your NetworkTrainer
model_filename = "best_fasterrcnn.pth" 
staging_folder = "fasterrcnn_output"       # We will create this folder to hold files
output_zip_name = "my_fasterrcnn_model"    # The name of the final zip file

# 2. Check if the model exists
if os.path.exists(model_filename):
    print(f"Found model: {model_filename}")
    
    # Create a clean directory to organize the files
    if os.path.exists(staging_folder):
        shutil.rmtree(staging_folder) # Clean up previous runs
    os.makedirs(staging_folder)
    
    # Copy the model weights into the folder
    print(f"Copying files to {staging_folder}...")
    shutil.copy(model_filename, os.path.join(staging_folder, model_filename))
    
    # (Optional) If you have other files like a log.txt or config.json, copy them here too:
    # if os.path.exists("training_log.txt"):
    #     shutil.copy("training_log.txt", os.path.join(staging_folder, "training_log.txt"))

    # 3. Zip the folder
    print(f"Zipping folder...")
    shutil.make_archive(output_zip_name, 'zip', root_dir=staging_folder)
    
    print(f"✅ Success! Created {output_zip_name}.zip")
    print("Click the link below to download your model:")
    
    # 4. Generate Download Link
    display(FileLink(f'{output_zip_name}.zip'))
    
else:
    print(f"❌ Error: Could not find '{model_filename}'.")
    print("Make sure your training loop finished successfully and the file exists in the file explorer.")

In [ ]:
# reload the trained model with correct num_classes
model = fasterrcnn_resnet50_fpn_v2(num_classes=4)
model.load_state_dict(torch.load("fasterrcnn_resnet50_fpn_v2.pth", map_location=device))
model.to(device)
model.eval()

In [ ]:
# reload the trained model with correct num_classes
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = fasterrcnn_resnet50_fpn_v2(num_classes=4)
model.load_state_dict(torch.load("/kaggle/input/resnet50-model/pytorch/default/1/resnet50-fasterrcnn-13.pth", map_location=device))
model.to(device)
model.eval()

In [ ]:
def compute_iou(gt_boxes, dt_boxes):
    """
    Compute IoU between all GT and detection boxes (vectorized).
    Boxes are in [x, y, w, h] format.
    Returns a (len(gt_boxes), len(dt_boxes)) matrix.
    """
    if len(gt_boxes) == 0 or len(dt_boxes) == 0:
        return np.zeros((len(gt_boxes), len(dt_boxes)))

    gt = gt_boxes.copy()
    dt = dt_boxes.copy()

    gt[:, 2:] += gt[:, :2]  # Convert [x,y,w,h] -> [x1,y1,x2,y2]
    dt[:, 2:] += dt[:, :2]

    gt_area = (gt[:, 2] - gt[:, 0]) * (gt[:, 3] - gt[:, 1])
    dt_area = (dt[:, 2] - dt[:, 0]) * (dt[:, 3] - dt[:, 1])

    inter_x1 = np.maximum(gt[:, None, 0], dt[None, :, 0])
    inter_y1 = np.maximum(gt[:, None, 1], dt[None, :, 1])
    inter_x2 = np.minimum(gt[:, None, 2], dt[None, :, 2])
    inter_y2 = np.minimum(gt[:, None, 3], dt[None, :, 3])

    inter_w = np.maximum(0, inter_x2 - inter_x1)
    inter_h = np.maximum(0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h

    union_area = gt_area[:, None] + dt_area[None, :] - inter_area
    iou = inter_area / (union_area + 1e-7)
    return iou


# --- (Combined, Efficient Evaluation Function) ---
def evaluate_model(model, data_loader, device, iou_threshold=0.5, score_threshold=0.5):
    """
    Runs model evaluation once and calculates BOTH mAP and fixed-threshold P/R.
    """
    model.eval()

    # --- Get COCO ground truth API ---
    base_dataset = data_loader.dataset
    if isinstance(base_dataset, torch.utils.data.Subset):
        base_dataset = base_dataset.dataset
    coco_gt = base_dataset.coco

    # --- 1. Variables for mAP ---
    coco_results = []
    
    # --- 2. Variables for manual P/R ---
    tp, fp, fn = 0, 0, 0

    with torch.no_grad():
        pbar = tqdm(data_loader, desc="Evaluating", leave=True)
        for imgs, targets in pbar:
            imgs = [img.to(device) for img in imgs]
            
            # --- RUN MODEL ONLY ONCE ---
            outputs = model(imgs)

            for output, target in zip(outputs, targets):
                image_id = int(target["image_id"].item())
                
                # Get raw model outputs
                boxes_raw = output["boxes"].cpu().numpy()    # [x1, y1, x2, y2]
                scores_raw = output["scores"].cpu().numpy()
                labels_raw = output["labels"].cpu().numpy()

                # ------------------------------------
                # PART A: Format results for mAP (COCOeval)
                # ------------------------------------
                for box, score, label in zip(boxes_raw, scores_raw, labels_raw):
                    x1, y1, x2, y2 = map(float, box)
                    coco_results.append({
                        "image_id": image_id,
                        "category_id": int(label),
                        "bbox": [x1, y1, float(x2 - x1), float(y2 - y1)], # [x, y, w, h]
                        "score": float(score)
                    })

                # ------------------------------------
                # PART B: Calculate fixed-threshold P/R
                # ------------------------------------
                
                # Get ground truth
                gt_ann_ids = coco_gt.getAnnIds(imgIds=image_id)
                gt_anns = coco_gt.loadAnns(gt_ann_ids)
                gt_boxes = np.array([ann['bbox'] for ann in gt_anns], dtype=np.float32) # [x, y, w, h]
                gt_labels = np.array([ann['category_id'] for ann in gt_anns], dtype=np.int64)

                # Filter detections by score
                keep = scores_raw >= score_threshold
                dt_boxes = boxes_raw[keep]  # [x1, y1, x2, y2]
                dt_scores = scores_raw[keep]
                dt_labels = labels_raw[keep]

                # Convert dt_boxes to [x, y, w, h] for compute_iou
                dt_boxes_coco = np.stack(
                    [dt_boxes[:, 0], dt_boxes[:, 1], dt_boxes[:, 2] - dt_boxes[:, 0], dt_boxes[:, 3] - dt_boxes[:, 1]],
                    axis=1
                )

                # --- Matching Logic (with bug fix) ---
                if len(gt_boxes) == 0:
                    fp += len(dt_boxes)
                    continue
                if len(dt_boxes) == 0:
                    fn += len(gt_boxes)
                    continue

                iou_matrix = compute_iou(gt_boxes, dt_boxes_coco)
                matched_gt = set()
                dt_order = np.argsort(dt_scores)[::-1]

                for dt_idx in dt_order:
                    best_gt_idx = -1
                    best_iou = iou_threshold  # Start with min IoU

                    for gt_idx in range(len(gt_boxes)):
                        if gt_idx in matched_gt:
                            continue
                        if gt_labels[gt_idx] != dt_labels[dt_idx]:
                            continue
                        
                        iou = iou_matrix[gt_idx, dt_idx]
                        if iou > best_iou:
                            best_iou = iou
                            best_gt_idx = gt_idx
                    
                    if best_gt_idx != -1:
                        tp += 1
                        matched_gt.add(best_gt_idx)
                    else:
                        fp += 1
                
                fn += len(gt_boxes) - len(matched_gt)

    # --- END OF LOOP ---
    
    all_metrics = {}

    # ------------------------------------
    # PART 1: Finalize mAP Calculation
    # ------------------------------------
    print("\n--- COCO mAP Results ---")
    if not coco_results:
        print("No detections found, skipping mAP calculation.")
    else:
        with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as f:
            json.dump(coco_results, f)
            result_file = f.name
        
        # Suppress pycocotools print statements
        old_stdout = sys.stdout
        sys.stdout = io.StringIO()
        
        coco_dt = coco_gt.loadRes(result_file)
        coco_eval = COCOeval(coco_gt, coco_dt, iouType="bbox")
        coco_eval.evaluate()
        coco_eval.accumulate()
        
        sys.stdout = old_stdout # Restore stdout
        
        # NOW print the summary and save all stats
        coco_eval.summarize()
        
        all_metrics.update({
            "mAP@50-95": float(coco_eval.stats[0]),
            "mAP@50": float(coco_eval.stats[1]),
            "mAP@75": float(coco_eval.stats[2]),
            "mAP_small": float(coco_eval.stats[3]),
            "mAP_medium": float(coco_eval.stats[4]),
            "mAP_large": float(coco_eval.stats[5]),
            "AR@1": float(coco_eval.stats[6]),
            "AR@10": float(coco_eval.stats[7]),
            "AR@100": float(coco_eval.stats[8]),
            "AR_small": float(coco_eval.stats[9]),
            "AR_medium": float(coco_eval.stats[10]),
            "AR_large": float(coco_eval.stats[11]),
        })

    # ------------------------------------
    # PART 2: Finalize P/R Calculation
    # ------------------------------------
    print(f"\n--- Metrics @ IoU={iou_threshold}, Score={score_threshold} ---")
    precision = tp / (tp + fp + 1e-7)
    recall = tp / (tp + fn + 1e-7)
    f1 = 2 * precision * recall / (precision + recall + 1e-7)
    
    print(f"TP: {tp}, FP: {fp}, FN: {fn}")
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")
    
    all_metrics.update({
        "Precision": float(precision),
        "Recall": float(recall),
        "F1": float(f1),
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn)
    })

    return all_metrics

In [ ]:
# Call the single evaluation function
all_results = evaluate_model(
    model, 
    test_loader, 
    device, 
    iou_threshold=0.5, 
    score_threshold=0.5
)

print("\n--- Summary ---")
print(f"mAP@50-95: {all_results.get('mAP@50-95', 0):.4f}")
print(f"mAP@50:    {all_results.get('mAP@50', 0):.4f}")
print(f"Precision:   {all_results.get('Precision', 0):.4f}")
print(f"Recall:    {all_results.get('Recall', 0):.4f}")

In [ ]:
MODEL_WEIGHTS_PATH = "/kaggle/input/resnet50-model/pytorch/default/1/resnet50-fasterrcnn-13.pth"
DATASET_ROOT = "BirdNest-4"

# Images are directly inside 'test'
TEST_IMG_DIR = os.path.join(DATASET_ROOT, "test")
# COCO JSON is inside 'test'
TEST_JSON_PATH = os.path.join(DATASET_ROOT, "test", "_annotations.coco.json")

NUM_IMAGES = 64
CONF_THRESHOLD = 0.25
NUM_CLASSES = 4 

# Visualization Config
GT_COLOR = (0, 255, 0)    # Green
PRED_COLOR = (0, 0, 255)  # Blue
TEXT_SCALE = 0.6
TEXT_THICKNESS = 1
BOX_THICKNESS = 2

CLASS_NAMES = {
    0: "background", 
    1: "Grade A",
    2: "Grade B",
    3: "Grade C"
}

In [ ]:
# --- 1. MODEL SETUP ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Loading Faster R-CNN model on {device}...")

model = fasterrcnn_resnet50_fpn_v2(num_classes=NUM_CLASSES)
if os.path.exists(MODEL_WEIGHTS_PATH):
    model.load_state_dict(torch.load(MODEL_WEIGHTS_PATH, map_location=device))
else:
    print("WARNING: Weights not found, using random initialization.")
model.to(device)
model.eval()

# --- 2. HELPER: LOAD COCO JSON ---
def load_coco_ground_truths(json_path):
    if not os.path.exists(json_path):
        print(f"Error: Annotation file not found at {json_path}")
        return {}

    print(f"Loading annotations from {json_path}...")
    with open(json_path, 'r') as f:
        data = json.load(f)

    img_id_to_name = {img['id']: img['file_name'] for img in data['images']}
    gt_dict = {}
    for ann in data['annotations']:
        img_id = ann['image_id']
        filename = img_id_to_name.get(img_id)
        
        if not filename: continue
        if filename not in gt_dict:
            gt_dict[filename] = []
            
        x, y, w, h = ann['bbox']
        cat_id = ann['category_id'] 
        
        x1, y1, x2, y2 = int(x), int(y), int(x + w), int(y + h)
        gt_dict[filename].append([x1, y1, x2, y2, cat_id])
        
    return gt_dict

# --- 3. DRAWING FUNCTION ---
def draw_boxes(image, gts, preds, class_names):
    vis_image = image.copy()
    
    # Draw GT (Green, Bottom-Right)
    for box in gts:
        x1, y1, x2, y2, cls_id = box
        label = f"GT: {class_names.get(cls_id, str(cls_id))}"
        cv2.rectangle(vis_image, (x1, y1), (x2, y2), GT_COLOR, BOX_THICKNESS)
        (w, h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, TEXT_SCALE, TEXT_THICKNESS)
        cv2.rectangle(vis_image, (x2 - w, y2 - h - 5), (x2, y2), GT_COLOR, -1)
        cv2.putText(vis_image, label, (x2 - w + 2, y2 - 5), cv2.FONT_HERSHEY_SIMPLEX, TEXT_SCALE, (0, 0, 0), TEXT_THICKNESS, cv2.LINE_AA)

    # Draw Preds (Blue, Top-Left)
    if preds:
        p_boxes = preds['boxes'].cpu().numpy()
        p_labels = preds['labels'].cpu().numpy()
        p_scores = preds['scores'].cpu().numpy()
        
        for box, label_id, score in zip(p_boxes, p_labels, p_scores):
            if score < CONF_THRESHOLD: continue
            x1, y1, x2, y2 = box.astype(int)
            label = f"Pred: {class_names.get(label_id, str(label_id))} ({score:.2f})"
            cv2.rectangle(vis_image, (x1, y1), (x2, y2), PRED_COLOR, BOX_THICKNESS)
            (w, h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, TEXT_SCALE, TEXT_THICKNESS)
            cv2.rectangle(vis_image, (x1, y1), (x1 + w, y1 + h + 5), PRED_COLOR, -1)
            cv2.putText(vis_image, label, (x1 + 2, y1 + h + 2), cv2.FONT_HERSHEY_SIMPLEX, TEXT_SCALE, (255, 255, 255), TEXT_THICKNESS, cv2.LINE_AA)
            
    return vis_image

In [ ]:
# --- MAIN EXECUTION ---

# A. Load GT Dictionary
coco_gts = load_coco_ground_truths(TEST_JSON_PATH)

# B. Find Images
print(f"Finding test images in {TEST_IMG_DIR}...")
image_patterns = [os.path.join(TEST_IMG_DIR, ext) for ext in ("*.jpg", "*.jpeg", "*.png")]
all_image_paths = []
for pattern in image_patterns:
    all_image_paths.extend(glob.glob(pattern))

if not all_image_paths:
    print(f"Error: No images found in {TEST_IMG_DIR}.")
else:
    num_to_sample = min(NUM_IMAGES, len(all_image_paths))
    print(f"Found {len(all_image_paths)} images. Sampling {num_to_sample}...")
    sampled_image_paths = random.sample(all_image_paths, num_to_sample)

    # Grid setup
    num_cols = 4
    num_rows = int(np.ceil(num_to_sample / num_cols))
    
    fig_w = num_cols * 4
    fig_h = num_rows * 4.2 
    
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(fig_w, fig_h))
    axes = axes.flatten()

    print("Running predictions...")

    for i, img_path in enumerate(sampled_image_paths):
        ax = axes[i]
        
        img_bgr = cv2.imread(img_path)
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        file_name = os.path.basename(img_path)
        gts = coco_gts.get(file_name, [])

        img_tensor = T.ToTensor()(img_rgb).to(device)
        with torch.no_grad():
            preds = model([img_tensor])[0]

        vis_image = draw_boxes(img_rgb, gts, preds, CLASS_NAMES)
        
        ax.imshow(vis_image)
        ax.axis('off')
        
        # --- MODIFICATION: Tiny text + Offset ---
        ax.text(0.5, -0.1, file_name, 
                transform=ax.transAxes, 
                ha='center', va='top', 
                fontsize=5, color='black', weight='bold') # <--- Reduced to 5

    # Turn off unused axes
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    # --- MODIFICATION: More breathing room between rows ---
    plt.subplots_adjust(hspace=0.5) # <--- Increased from 0.3
    
    print("Displaying plot...")
    plt.show()
    fig.savefig("fasterrcnn_coco_grid_named.png")

In [ ]:
print("Hello")